# 教程 03：机器学习分子动力学

**学习目标** ⭐
- 了解分子动力学 (MD) 模拟的基本流程
- 学习 Lennard-Jones 力场的计算逻辑
- 掌握本仓库 AI4MD 算子的调用方式

---

## 分子动力学模拟流程

分子动力学模拟的基本循环：

```
1. 初始化原子坐标和速度
2. 计算原子间力 (力场 → 力)
3. 更新速度和位置 (时间积分器)
4. 应用约束 (如 SHAKE)
5. 计算测量量 (温度、压力、能量)
6. 重复 2-5 至模拟结束
```

本仓库在 AI4MD 目录下实现了完整的 MD 算子链：

| 算子 | 功能 | 物理量 |
|------|------|--------|
| L-J | 范德华力计算 | 非键相互作用 |
| GAFF2 | 通用 Amber 力场 | 键合 + 非键 |
| PME | 长程静电 | 周期性静电 |
| SHAKE | 键长约束 | 约束算法 |
| Velocity Verlet | 时间积分 | 位置/速度更新 |
| DPD | 耗散粒子动力学 | 介观模拟 |

In [ ]:
import os

repo_root = os.path.abspath(os.path.join(os.getcwd(), '..'))
lj_dir = os.path.join(repo_root, 'simulation', 'AI4MD', 'Lennard_Jones')

print("LJ 算子文件:")
for f in sorted(os.listdir(os.path.join(lj_dir, 'op_kernel'))):
    print(f"  📄 {f}")
for f in sorted(os.listdir(os.path.join(lj_dir, 'op_host'))):
    print(f"  📄 {f}")

## LJ 力场原理

Lennard-Jones 势能函数描述了中性原子间的范德华相互作用：

$$E_{LJ}(r) = 4\epsilon\left[\left(\frac{\sigma}{r}\right)^{12} - \left(\frac{\sigma}{r}\right)^6\right]$$

其中 $r$ 为原子间距，$\epsilon$ 为势阱深度，$\sigma$ 为势能零点对应的距离。

对应的力为负梯度：

$$F_{LJ}(r) = -\frac{dE}{dr} = 24\epsilon\left[2\left(\frac{\sigma}{r}\right)^{12} - \left(\frac{\sigma}{r}\right)^6\right]\frac{1}{r}$$

In [ ]:
import torch
import numpy as np

def lj_force_pytorch(positions, epsilon=1.0, sigma=1.0, cutoff=2.5):
    """纯 PyTorch 实现的 LJ 力计算 (参考实现)"""
    n_atoms = positions.shape[0]
    forces = torch.zeros_like(positions)
    energy = 0.0

    for i in range(n_atoms):
        for j in range(i + 1, n_atoms):
            dr = positions[i] - positions[j]
            r = torch.norm(dr)
            if r < cutoff * sigma:
                sr6 = (sigma / r) ** 6
                sr12 = sr6 ** 2
                e_lj = 4.0 * epsilon * (sr12 - sr6)
                energy = energy + e_lj
                f_mag = 24.0 * epsilon * (2.0 * sr12 - sr6) / r
                f_vec = f_mag * dr / r
                forces[i] = forces[i] + f_vec
                forces[j] = forces[j] - f_vec

    return forces, energy

# 测试：氩原子二聚体
pos = torch.tensor([[0.0, 0.0, 0.0], [2.0, 0.0, 0.0]], dtype=torch.float32)
f, e = lj_force_pytorch(pos, epsilon=0.0103, sigma=3.40)

print(f"氩原子二聚体势能: {e.item():.6f} eV")
print(f"原子1受力: ({f[0,0]:.4f}, {f[0,1]:.4f}, {f[0,2]:.4f}) eV/Å")
print(f"原子2受力: ({f[1,0]:.4f}, {f[1,1]:.4f}, {f[1,2]:.4f}) eV/Å")

## LJ 势能曲线

画出 LJ 势能和力随距离的变化曲线，理解其物理行为。

In [ ]:
try:
    import matplotlib.pyplot as plt

    r = torch.linspace(0.8, 3.0, 100) * 3.4  # 以 σ 为单位
    sr6 = (3.40 / r) ** 6
    sr12 = sr6 ** 2
    e = 4.0 * 0.0103 * (sr12 - sr6)
    f = 24.0 * 0.0103 * (2.0 * sr12 - sr6) / r

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

    ax1.plot(r.numpy(), e.numpy(), 'b-', linewidth=2)
    ax1.axhline(0, color='gray', linestyle='--', alpha=0.5)
    ax1.axvline(3.4 * 2**(1/6), color='r', linestyle='--', alpha=0.5, label='r₀ = 2^(1/6)σ')
    ax1.set_xlabel('r (Å)')
    ax1.set_ylabel('势能 (eV)')
    ax1.set_title('LJ 势能曲线')
    ax1.legend()
    ax1.grid(alpha=0.3)

    ax2.plot(r.numpy(), f.numpy(), 'r-', linewidth=2)
    ax2.axhline(0, color='gray', linestyle='--', alpha=0.5)
    ax2.set_xlabel('r (Å)')
    ax2.set_ylabel('力 (eV/Å)')
    ax2.set_title('LJ 力曲线')
    ax2.grid(alpha=0.3)

    plt.tight_layout()
    plt.show()
except ImportError:
    print("需要 matplotlib")

## MD 模拟循环

下面演示一个完整的 MD 时间步：先计算力，再用 Velocity Verlet 更新坐标和速度。

In [ ]:
def velocity_verlet_step(pos, vel, forces, mass, dt, compute_force_fn):
    """单步 Velocity Verlet 积分"""
    # 半部速度更新
    vel_half = vel + 0.5 * dt * forces / mass
    # 位置更新
    pos_new = pos + dt * vel_half
    # 新力计算
    forces_new, _ = compute_force_fn(pos_new)
    # 后半个速度更新
    vel_new = vel_half + 0.5 * dt * forces_new / mass
    return pos_new, vel_new, forces_new

# 4 个氩原子的测试
n_atoms = 4
pos = torch.randn(n_atoms, 3) * 2.0
vel = torch.randn(n_atoms, 3) * 0.1
mass = 39.95  # 氩的原子质量 (amu)
dt = 0.001  # 时间步长 (ps)

print(f"初始位置:\n{pos}\n")
print("执行 10 个 MD 步...")
for step in range(10):
    forces, energy = lj_force_pytorch(pos, epsilon=0.0103, sigma=3.40)
    pos, vel, forces = velocity_verlet_step(pos, vel, forces, mass, dt, lambda p: lj_force_pytorch(p))
    if step % 5 == 0:
        ke = 0.5 * mass * (vel ** 2).sum().item()
        pe = energy.item()
        print(f"  步 {step}: PE={pe:.4f}, KE={ke:.4f}, E={pe+ke:.4f}")
print("MD 模拟完成")

## 总结 📋

- LJ 力场是最基础的 MD 势能函数，$E \propto \epsilon[(\sigma/r)^{12} - (\sigma/r)^6]$
- 本仓库提供了 LJ/GAFF2/PME/SHAKE/VV/DPD 等 6 个 MD 相关 Ascend C 算子
- MD 模拟的基本循环：力计算 → 积分器 → 约束 → 采样
- PyTorch 参考实现可在无 NPU 环境下学习和验证

## 思考练习 ✏️

1. 尝试修改 LJ 参数 $\epsilon$ 和 $\sigma$，观察势能曲线如何变化
2. 扩展系统到 100 个原子，对比纯 PyTorch 和 NumPy 实现的速度
3. 阅读本仓库的 Ascend C LJ 算子代码，理解 NPU 如何加速对力计算